# Serializadores customizados com tipos `Annotated`

Já vimos como customizar a serialização anexando o serializador a campos específicos com o decorador `@field_serializer`.

Com validadores, podemos anexar o validador a um campo (ou a vários) com um decorador, **ou** anexá-lo a um tipo `Annotated`.

Com serializadores é igual: além do decorador no campo, podemos anexar um serializador customizado a um tipo `Annotated`.

Suponha que queremos um tipo de datetime que:
- desserialize outros formatos de string de datetime (como `1/1/2020 3pm`);
- sempre armazene o datetime como **aware em UTC**;
- serialize para um objeto `datetime` quando o destino for um `dict` Python;
- serialize para JSON neste formato: `YYYY/MM/DD HH:MM AM/PM (UTC)`.

Já vimos como fazer tudo isso com decoradores ou com tipos `Annotated`. Começamos pelos validadores, anexados a um tipo `Annotated`.

Nada de novo aqui:

In [1]:
from datetime import datetime
from typing import Annotated, Any

import pytz
from dateutil.parser import parse

from pydantic import BaseModel, AfterValidator, BeforeValidator


def make_utc(dt: datetime) -> datetime:
    if dt.tzinfo is None:
        dt = pytz.utc.localize(dt)
    else:
        dt = dt.astimezone(pytz.utc)
    return dt

def parse_datetime(value: Any):
    if isinstance(value, str):
        try:
            return parse(value)
        except Exception as ex:
            raise ValueError(str(ex))
    return value


DateTimeUTC = Annotated[datetime, BeforeValidator(parse_datetime), AfterValidator(make_utc)]

> 📝 **Nota de revisão:** desde o Python 3.9 dá para fazer o mesmo sem `pytz`, só com a biblioteca padrão:
> `dt.replace(tzinfo=timezone.utc)` (naive) e `dt.astimezone(timezone.utc)` (aware), com `from datetime import timezone`.
> O `pytz` é necessário apenas para código legado; o resultado aqui é equivalente:

In [2]:
from datetime import timezone

def make_utc_stdlib(dt: datetime) -> datetime:
    return dt.replace(tzinfo=timezone.utc) if dt.tzinfo is None else dt.astimezone(timezone.utc)

naive = datetime(2020, 1, 1, 15)
print(f"make_utc(naive) == make_utc_stdlib(naive): {make_utc(naive) == make_utc_stdlib(naive)}")

make_utc(naive) == make_utc_stdlib(naive): True


E podemos usar esse tipo `Annotated` em qualquer modelo:

In [3]:
class Model(BaseModel):
    dt: DateTimeUTC

Mas ainda falta o serializador customizado para JSON.

Primeiro, do jeito que já aprendemos: com um decorador.

Criamos uma função que devolve o datetime formatado como string:

In [4]:
def dt_json_serializer(dt: datetime) -> str:
    return dt.strftime("%Y/%m/%d %I:%M %p UTC")

In [5]:
print(f"dt_json_serializer(datetime(2020,1,1,15,0,0)): {dt_json_serializer(datetime(2020,1,1,15,0,0))!r}")

dt_json_serializer(datetime(2020,1,1,15,0,0)): '2020/01/01 03:00 PM UTC'


> 📝 **Nota de revisão:** o requisito pede `(UTC)` entre parênteses, mas o formato usado no curso produz `UTC` sem parênteses.
> Mantivemos o formato original nesta aula; na prática, o formato certo é `"%Y/%m/%d %I:%M %p (UTC)"`.

Agora anexamos ao campo do modelo.

Só precisamos disso na serialização para **JSON**, e só quando o campo não for `None` — então definimos o serializador assim:

In [6]:
from pydantic import field_serializer

In [7]:
class Model(BaseModel):
    dt: DateTimeUTC

    @field_serializer("dt", when_used="json-unless-none")
    def serialize_dt_to_json(self, value: datetime) -> str:
        return dt_json_serializer(value)

Testando o modelo:

In [8]:
m = Model(dt="2020/1/1 3pm")
print(f"m: {m!r}")

m: Model(dt=datetime.datetime(2020, 1, 1, 15, 0, tzinfo=<UTC>))


E serializando:

In [9]:
print(f"m.model_dump(): {m.model_dump()!r}")

m.model_dump(): {'dt': datetime.datetime(2020, 1, 1, 15, 0, tzinfo=<UTC>)}


In [10]:
print(f"m.model_dump_json(): {m.model_dump_json()!r}")

m.model_dump_json(): '{"dt":"2020/01/01 03:00 PM UTC"}'


Funciona como queríamos.

Mas esse serializador poderia ser anexado ao **tipo** `Annotated` em vez de ao campo específico do modelo.

Para isso usamos o `PlainSerializer` do Pydantic.

Um *plain serializer* substitui por completo o serializador do Pydantic pelo nosso (que é o que fazíamos com o decorador).

In [11]:
from pydantic import PlainSerializer

In [12]:
DateTimeUTC = Annotated[
    datetime,
    BeforeValidator(parse_datetime),
    AfterValidator(make_utc),
    PlainSerializer(dt_json_serializer),
]

Só falta dizer **quando** o serializador deve ser usado — no nosso caso, `json-unless-none`.

Basta passar isso como argumento nomeado:

In [13]:
DateTimeUTC = Annotated[
    datetime,
    BeforeValidator(parse_datetime),
    AfterValidator(make_utc),
    PlainSerializer(dt_json_serializer, when_used="json-unless-none"),
]

Agora temos um tipo totalmente reutilizável, com validadores e serializador customizados.

In [14]:
class Model(BaseModel):
    dt: DateTimeUTC

Testando:

In [15]:
m = Model(dt="2020/1/1 3pm")
print(f"m: {m!r}")

m: Model(dt=datetime.datetime(2020, 1, 1, 15, 0, tzinfo=<UTC>))


In [16]:
print(f"m.model_dump(): {m.model_dump()!r}")

m.model_dump(): {'dt': datetime.datetime(2020, 1, 1, 15, 0, tzinfo=<UTC>)}


In [17]:
print(f"m.model_dump_json(): {m.model_dump_json()!r}")

m.model_dump_json(): '{"dt":"2020/01/01 03:00 PM UTC"}'


> 📝 **Nota de revisão — complementos (não cobertos na aula):**
>
> - **Opcional funciona de graça:** `DateTimeUTC | None` serializa `None` como `null`, sem chamar o serializador
>   (é o que o `"json-unless-none"` garante).
> - **`WrapSerializer`** é a contraparte do `WrapValidator`: recebe também o `handler` do Pydantic, útil para pós-processar
>   a serialização padrão em vez de substituí-la.
> - O **schema JSON de serialização** reflete o tipo de retorno anotado da função (`str`), por isso anotar `-> str` importa.

In [18]:
from pydantic import WrapSerializer

class OptionalModel(BaseModel):
    dt: DateTimeUTC | None = None

print(f"OptionalModel().model_dump_json(): {OptionalModel().model_dump_json()}")

UpperStr = Annotated[str, WrapSerializer(lambda value, handler: handler(value).upper())]

class Tagged(BaseModel):
    tag: UpperStr

print(f"Tagged(tag='beta').model_dump(): {Tagged(tag='beta').model_dump()}")
print(f"schema de serialização de dt: {Model.model_json_schema(mode='serialization')['properties']['dt']}")

OptionalModel().model_dump_json(): {"dt":null}
Tagged(tag='beta').model_dump(): {'tag': 'BETA'}
schema de serialização de dt: {'title': 'Dt', 'type': 'string'}
